# 00 Environment and Dataset Validation

Use this notebook to confirm the runtime, package imports, raw-data path, and expected table availability before moving to EDA.

This notebook stays manual by design. It does not copy or download data automatically.

In [ ]:
from pathlib import Path
import platform
import sys

try:
    import google.colab  # type: ignore  # noqa: F401
    RUNTIME = "colab"
except ImportError:
    RUNTIME = "local"

root = next(
    (candidate for candidate in [Path.cwd(), *Path.cwd().parents] if (candidate / "pyproject.toml").exists()),
    Path.cwd(),
)
src = root / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

from credit_visable.utils.paths import get_paths

paths = get_paths()
DEFAULT_RAW_DATA_DIR = paths.data_raw

print(f"Runtime: {RUNTIME}")
print(f"Python version: {platform.python_version()}")
print(f"Current working directory: {Path.cwd()}")
print(f"Resolved project root: {paths.root}")
print(f"Default raw data directory: {DEFAULT_RAW_DATA_DIR}")


In [ ]:
import importlib

import pandas as pd
from IPython.display import display

required_modules = ["numpy", "pandas", "sklearn", "matplotlib", "seaborn", "yaml"]
rows = []
for module_name in required_modules:
    try:
        importlib.import_module(module_name)
        rows.append({"module": module_name, "status": "ok", "detail": "import succeeded"})
    except Exception as exc:
        rows.append(
            {
                "module": module_name,
                "status": "failed",
                "detail": f"{exc.__class__.__name__}: {exc}",
            }
        )

import_status = pd.DataFrame(rows)
display(import_status)

failed_modules = import_status.loc[import_status["status"] != "ok", "module"].tolist()
if failed_modules:
    print("Phase 0 is blocked until the missing imports are fixed.")
    print(f"Missing or failing imports: {failed_modules}")
else:
    print("All required Phase 0 imports are available.")


In [ ]:
from credit_visable.config import load_settings

settings = load_settings()

# Edit only this variable when your local CSV files live somewhere else.
RAW_DATA_DIR = DEFAULT_RAW_DATA_DIR
# Colab example:
# RAW_DATA_DIR = Path("/content/drive/MyDrive/home-credit/data/raw")

print(f"Using raw data directory: {RAW_DATA_DIR}")
print(f"Configured target column: {settings.target_column}")
print(f"Configured customer key: {settings.id_column}")


In [ ]:
from credit_visable.data import summarize_table_availability

table_summary = summarize_table_availability(data_dir=RAW_DATA_DIR, settings=settings)
display(table_summary)

available_count = int(table_summary["available"].sum())
expected_count = int(len(table_summary))
print(f"Available tables: {available_count}/{expected_count}")

if available_count == 0:
    print("No expected raw tables were found in the selected raw-data directory.")
    print("Manual next steps:")
    print("- Copy the expected CSV files into data/raw/")
    print(' - Colab path override: RAW_DATA_DIR = Path("/content/drive/MyDrive/home-credit/data/raw")')
elif not bool(table_summary.loc[table_summary["table_name"] == "application_train", "available"].item()):
    print("application_train.csv is still missing, so do not continue to EDA yet.")
    print("Manual next steps:")
    print("- Copy application_train.csv into data/raw/")
    print(' - Colab path override: RAW_DATA_DIR = Path("/content/drive/MyDrive/home-credit/data/raw")')
else:
    print("application_train.csv is available. Partial table coverage is acceptable for Phase 0.")


In [ ]:
from credit_visable.data import load_application_train

application_train_available = bool(
    table_summary.loc[table_summary["table_name"] == "application_train", "available"].item()
)

if application_train_available:
    application_train_preview = load_application_train(data_dir=RAW_DATA_DIR, nrows=5)
    display(application_train_preview)
    print("Phase 0 is complete. Continue to notebooks/01_eda.ipynb once this preview looks correct.")
else:
    print("Phase 0 stop: application_train.csv must exist before you move to notebooks/01_eda.ipynb.")
